# 03 · 문서를 AI Search에 넣기 · 50분

## 학습 목표
- 문서 → 청크 → 임베딩 → 인덱스 → 업로드 단계를 구분합니다.
- 출처 메타데이터와 1536차원 벡터 필드를 확인합니다.

## 개념
임베딩은 텍스트를 유사도 검색용 숫자 벡터로 바꿉니다. 검색 인덱스에 넣는 것과 LLM을 학습시키는 것은 다릅니다. **검색 가능하게 저장하는 작업**입니다.

`data/policies/`의 문서는 이 실습용으로 새로 작성한 가상 한빛테크 규정입니다. 실제 사내 문서·개인정보를 넣지 마세요. 출처는 파일명·절·청크 ID를 보존합니다. Markdown의 `page=1`은 논리 페이지이며 실제 PDF 쪽수가 아닙니다.

⚠️ 이 노트북은 **본인 인덱스를 생성/갱신하고 문서를 업로드**합니다. 개인 키의 서버 측 권한은 본인 인덱스로 제한됩니다.

In [ ]:
from pathlib import Path
import sys
import json
from dotenv import load_dotenv

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from hanbit_assistant.config import Settings
from hanbit_assistant.clients import create_openai_client
settings = Settings.from_env()
# 비밀 키, 환경변수 전체, HTTP 헤더를 출력하지 않습니다.
print({"model": settings.model, "my_index": settings.search_index})


In [ ]:
from hanbit_assistant.ingestion import load_policy_chunks, chunk_markdown
chunks = load_policy_chunks(ROOT / "data/policies")
print("문서 수:", len({c["source"] for c in chunks}), "청크 수:", len(chunks))
print(json.dumps(chunks[2], ensure_ascii=False, indent=2))
assert len(chunks) > 0


## 실험 · 예측 → 실행 → 관찰 → 변경
**예측:** 한 문장을 더 작은 청크로 자르면 “JPY 18,000”과 “세금 포함”이 함께 남을까요?
**실행:** 아래 `MAX_CHARS=100`, `OVERLAP=15`로 로컬 청킹을 해 봅니다. 여기까지 업로드나 모델 호출은 없습니다.
**관찰:** `일본 숙박` 절을 읽으며 답에 필요한 두 조건이 같은 청크에 있는지 표시합니다.
**변경:** `MAX_CHARS=300`만 바꾸고 이 셀을 다시 실행합니다. 청크 수와 조건 보존을 비교합니다.
그 뒤 선택 실험으로 `OVERLAP`만 바꿉니다. 작게 자르는 것이 항상 더 좋은 것은 아닙니다.

아래는 메모리의 `small_chunks`만 바꾸며 **업로드는 원래 `chunks`**를 사용합니다.
원문·인덱스 차원을 바꾸는 실험이 아닙니다. 실험 청크를 기본 인덱스에 섞지 마세요.


In [ ]:
prediction = ""  # 청크 수와 조건 보존을 예측
MAX_CHARS = 100  # 변경 실험: 300
OVERLAP = 15
text = (ROOT / "data/policies/02_lodging.md").read_text(encoding="utf-8")
small_chunks = chunk_markdown(text, "data/policies/02_lodging.md", max_chars=MAX_CHARS, overlap=OVERLAP)
print("기본 청크 수:", len([c for c in chunks if c["source"].endswith("02_lodging.md")]))
print("실험 청크 수:", len(small_chunks))
for part in small_chunks:
    if part["section"] == "일본 숙박":
        print(part["id"][:12], len(part["content"]), part["content"])
assert all({"id", "source", "section", "page", "content"} <= c.keys() for c in small_chunks)


### 2. 임베딩과 인덱스 스키마
한 쿼리 벡터와 문서 벡터는 **같은 모델·차원**을 사용해야 합니다. HNSW는 가까운 벡터를 빠르게 찾기 위한 알고리즘입니다. 인덱스에는 한국어 키워드 검색 필드와 벡터 필드를 함께 둡니다.
**미니 실험:** 아래 세 문장 중 첫 문장과 더 가까울 쌍을 예측한 후 실제 임베딩 유사도를 비교합니다.
세 번째 문장은 대조군입니다. 유사도는 사실 여부/규정 적용 여부 점수가 아닙니다. 숫자 벡터를 사람이 문장처럼 읽지는 않습니다.


In [ ]:
from math import sqrt
from hanbit_assistant.search import build_index, embed_texts, PolicySearch
texts = ["도쿄 호텔 비용 한도", "일본 출장 숙박비", "노트북 비밀번호 변경"]
async with create_openai_client(settings) as client:
    example_vectors = await embed_texts(client, texts, model=settings.embedding_model)
print("실제 벡터 차원:", [len(v) for v in example_vectors])
for i in (1, 2):
    a, b = example_vectors[0], example_vectors[i]
    similarity = sum(x * y for x, y in zip(a, b)) / (sqrt(sum(x*x for x in a)) * sqrt(sum(y*y for y in b)))
    print(texts[0], "↔", texts[i], "코사인 유사도:", round(similarity, 4))
print([(f.name, f.type) for f in build_index(settings.search_index).fields])
assert all(len(v) == 1536 for v in example_vectors)


### 3. 생성·업로드·실제 읽기 검증
리소스(Search 서비스)는 강사가 미리 만듭니다. 학생은 서비스 안의 **자기 인덱스**만 다룹니다. `create_index()`는 인덱스 구조, `upload_chunks()`는 임베딩 생성과 문서 업로드입니다.
**쓰기 전 체크포인트:** 출력된 `my_index`가 본인 배정값인지 확인합니다.
이 셀 실행은 본인 인덱스 생성/갱신·임베딩 비용·업로드를 수반합니다. 반영 후 개수와 검색 결과를 실제로 다시 읽습니다.
실험값을 바꿀 때는 앞 청킹 셀만 다시 실행하세요. 전체 Run All은 업로드도 다시 실행합니다.


In [ ]:
service = PolicySearch(settings)
try:
    await service.create_index()
    uploaded = await service.upload_chunks(chunks)  # 기본 원문만 업로드
    assert uploaded == len(chunks), "업로드 응답 수가 예상과 다릅니다."
    print({"uploaded": uploaded, "index": settings.search_index})
finally:
    await service.close()


### 4. 읽기 전용 확인 · 반영이 늦으면 이 셀만 재실행
아래 셀은 새 클라이언트로 실제 개수/검색 결과를 읽으며 인덱스를 생성하거나 문서를 다시 업로드하지 않습니다.
`indexed_count`가 기본 청크 수보다 많으면 옛 데이터가 남았을 수 있으므로 강사에게 알립니다. 임의로 삭제하지 않습니다.


In [ ]:
import asyncio
service = PolicySearch(settings)
try:
    for attempt in range(30):
        count = await service.client.get_document_count()
        if count >= len(chunks):
            break
        await asyncio.sleep(1)
    assert count >= len(chunks), "검색 반영이 지연됩니다. 잠시 후 이 읽기 확인 셀만 재실행하세요."
    print({"expected_chunks": len(chunks), "indexed_count": count, "index": settings.search_index})
    if count != len(chunks):
        print("체크포인트 주의: 예상과 실제 개수가 다릅니다. 옛 청크 여부를 강사와 확인하세요.")
    hits = await service.search("도쿄 숙박비", mode="keyword", top=3)
    print([(h["title"], h["section"]) for h in hits])
    assert hits, "개수는 있지만 검색 결과가 없습니다. 성공으로 간주하지 마세요."
finally:
    await service.close()


## 관찰 기록과 체크포인트
| MAX_CHARS / OVERLAP | 청크 수 | 금액과 세금 조건이 함께 있는가? |
|---|---|---|
| 100 / 15 | 직접 기록 | 직접 기록 |
| 300 / 15 | 직접 기록 | 직접 기록 |

- [ ] 벡터 차원은 1536이며 문서와 질의에 같은 모델/차원을 쓴다.
- [ ] `uploaded`, 실제 `indexed_count`, 검색한 절을 확인했다. 숫자가 예상보다 많으면 강사에게 알린다.
- [ ] 청크에서 `source`와 `section`으로 원문을 찾았다.

**결과 질문:** 인덱스에 자료를 추가한 뒤 LLM의 가중치가 바뀌었나요? 다른 사람이 접근해도 되나요?
**연습:** 출처 메타데이터가 없을 때 검증할 수 없는 항목을 두 개 적고, 업로드 코드가 `chunks`와 `small_chunks` 중 무엇을 쓰는지 찾으세요.

<details><summary>참고 풀이 · 예시 해설 (관찰 후 열기)</summary>
인덱싱은 외부 검색 저장소에 넣는 작업이지 LLM 재학습이 아닙니다. 출처가 없으면 원문 위치·적용 조건 확인이 어렵습니다.
작은 청크는 조건을 분리할 수 있고 겹침은 단절을 줄이지만 저장량이 늘 수 있습니다. 기본 업로드는 `chunks`입니다.
같은 원문은 같은 ID로 upsert되지만 원문 변경 후 옛 청크까지 삭제하는 전체 동기화는 아닙니다.
접근 권한은 본인 APIM 키의 서버 측 인덱스 제한으로 결정되며 인덱스 이름 변경으로 권한이 생기지 않습니다.
</details>

### 실패하면
400 차원 불일치: 모델/1536차원 설정을 확인하고 임의로 새 인덱스를 만들지 않습니다.
403: 본인 키/인덱스 배정 확인. 개수 반영 지연: 잠시 후 읽기 확인 셀을 재실행하되 불필요한 반복 업로드는 피합니다.
업로드 일부 실패/413: 중단하고 강사에게 배치/입력 크기를 확인받습니다. 실패를 성공으로 기록하지 않습니다.
기본 문서는 수정하지 않으며 다른 학생 인덱스 또는 공용 `/openai` 설정을 건드리지 않습니다.
